In [4]:
from pathlib import Path
import json
import re
import unicodedata
from collections import Counter, defaultdict

import numpy as np
import pandas as pd


# =========================================================
# 0) CONFIGURACIÓN PARA NOTEBOOK
# =========================================================

if (Path.cwd() / "datasets").exists():
    BASE_DIR = Path.cwd()
elif (Path.cwd().parent / "datasets").exists():
    BASE_DIR = Path.cwd().parent
else:
    raise FileNotFoundError(
        "No se encontró la carpeta 'datasets' desde el directorio actual ni desde su padre."
    )

DATA_DIR = BASE_DIR / "datasets"
OUT_DIR = BASE_DIR / "Documentation" / "eda_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FGJ_FILE = DATA_DIR / "carpetasFGJ_acumulado_2025_01.csv"
POBREZA_FILE = DATA_DIR / "enigh_16_20.csv"

FGJ_CHUNKSIZE = 200_000

print("BASE_DIR:", BASE_DIR)
print("FGJ_FILE:", FGJ_FILE, "| Existe:", FGJ_FILE.exists())
print("POBREZA_FILE:", POBREZA_FILE, "| Existe:", POBREZA_FILE.exists())
print("OUT_DIR:", OUT_DIR)


# =========================================================
# 1) UTILIDADES GENERALES
# =========================================================

def normalize_text(value):
    if pd.isna(value):
        return None
    text = str(value).strip().upper()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^A-Z0-9]+", " ", text)
    text = " ".join(text.split())
    if not text:
        return None
    return text


ALCALDIA_ALIASES = {
    "GUSTAVO A MADERO": "GUSTAVO A MADERO",
    "GUSTAVO A MADERO ": "GUSTAVO A MADERO",
    "MAGDALENA CONTRERAS": "LA MAGDALENA CONTRERAS",
    "ALVARO OBREGON": "ALVARO OBREGON",
    "MIGUEL HIDALGO": "MIGUEL HIDALGO",
    "VENUSTIANO CARRANZA": "VENUSTIANO CARRANZA",
    "BENITO JUAREZ": "BENITO JUAREZ",
    "CUAJIMALPA DE MORELOS": "CUAJIMALPA DE MORELOS",
    "CUAUHTEMOC": "CUAUHTEMOC",
    "COYOACAN": "COYOACAN",
    "AZCAPOTZALCO": "AZCAPOTZALCO",
    "IZTACALCO": "IZTACALCO",
    "IZTAPALAPA": "IZTAPALAPA",
    "MILPA ALTA": "MILPA ALTA",
    "TLAHUAC": "TLAHUAC",
    "TLALPAN": "TLALPAN",
    "XOCHIMILCO": "XOCHIMILCO",
}

MAP_MUNICIPIO_CDMX = {
    2: "AZCAPOTZALCO",
    3: "COYOACAN",
    4: "CUAJIMALPA DE MORELOS",
    5: "GUSTAVO A MADERO",
    6: "IZTACALCO",
    7: "IZTAPALAPA",
    8: "LA MAGDALENA CONTRERAS",
    9: "MILPA ALTA",
    10: "ALVARO OBREGON",
    11: "TLAHUAC",
    12: "TLALPAN",
    13: "XOCHIMILCO",
    14: "BENITO JUAREZ",
    15: "CUAUHTEMOC",
    16: "MIGUEL HIDALGO",
    17: "VENUSTIANO CARRANZA",
}

def normalize_alcaldia(value):
    txt = normalize_text(value)
    if txt is None:
        return None
    return ALCALDIA_ALIASES.get(txt, txt)

def norm_col(col):
    col = normalize_text(col)
    return col if col is not None else ""

def resolve_column(columns, aliases):
    cols_map = {norm_col(c): c for c in columns}
    for alias in aliases:
        alias_norm = norm_col(alias)
        if alias_norm in cols_map:
            return cols_map[alias_norm]
    return None

def save_csv(df, filename):
    path = OUT_DIR / filename
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"[OK] Guardado CSV: {path}")

def save_json(obj, filename):
    path = OUT_DIR / filename
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    print(f"[OK] Guardado JSON: {path}")

def save_txt(text, filename):
    path = OUT_DIR / filename
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"[OK] Guardado TXT: {path}")

def weighted_mean(series, weights):
    mask = series.notna() & weights.notna()
    if mask.sum() == 0:
        return np.nan
    s = pd.to_numeric(series[mask], errors="coerce")
    w = pd.to_numeric(weights[mask], errors="coerce")
    mask2 = s.notna() & w.notna()
    if mask2.sum() == 0 or w[mask2].sum() == 0:
        return np.nan
    return float(np.average(s[mask2], weights=w[mask2]))

def weighted_mode(series, weights):
    mask = series.notna() & weights.notna()
    if mask.sum() == 0:
        return np.nan
    tmp = (
        pd.DataFrame({"valor": series[mask], "peso": weights[mask]})
        .groupby("valor", dropna=False)["peso"]
        .sum()
        .sort_values(ascending=False)
    )
    if tmp.empty:
        return np.nan
    return tmp.index[0]


# =========================================================
# 2) LECTURA ROBUSTA DE CSV
# =========================================================

ENCODINGS_TO_TRY = ["utf-8", "utf-8-sig", "cp1252", "latin1"]

def read_csv_flexible(path, nrows=None, usecols=None, chunksize=None, low_memory=False):
    last_error = None
    for enc in ENCODINGS_TO_TRY:
        try:
            df = pd.read_csv(
                path,
                encoding=enc,
                nrows=nrows,
                usecols=usecols,
                chunksize=chunksize,
                low_memory=low_memory,
            )
            print(f"[OK] Archivo leído con encoding: {enc}")
            return df, enc
        except Exception as e:
            last_error = e
    raise last_error

def read_header(path):
    header_df, enc = read_csv_flexible(path, nrows=0)
    return header_df.columns.tolist(), enc


# =========================================================
# 3) ANÁLISIS FGJ
# =========================================================

def analyze_fgj(path):
    print("\n" + "=" * 90)
    print("ANÁLISIS FGJ")
    print("=" * 90)

    cols, enc = read_header(path)

    # Resolver columnas clave de forma robusta
    col_anio_hecho = resolve_column(cols, ["anio_hecho", "año_hecho"])
    col_anio_inicio = resolve_column(cols, ["anio_inicio", "año_inicio"])
    col_delito = resolve_column(cols, ["delito"])
    col_categoria = resolve_column(cols, ["categoria_de_competencia", "categoría_de_competencia"])
    col_alcaldia_hecho = resolve_column(cols, ["alcaldia_hech", "alcaldia_hecho"])
    col_alcaldia_cat = resolve_column(cols, ["alcaldia_cata", "alcaldia_cat"])
    col_municipio = resolve_column(cols, ["municipio_he", "municipio_h", "municipio"])
    col_latitud = resolve_column(cols, ["latitud"])
    col_longitud = resolve_column(cols, ["longitud"])
    col_fecha_hecho = resolve_column(cols, ["fecha_hecho"])
    col_fecha_inicio = resolve_column(cols, ["fecha_inicio"])

    fgj_meta = {
        "encoding_usado": enc,
        "n_columnas": len(cols),
        "columnas": cols,
        "columnas_resueltas": {
            "anio_hecho": col_anio_hecho,
            "anio_inicio": col_anio_inicio,
            "delito": col_delito,
            "categoria_de_competencia": col_categoria,
            "alcaldia_hecho": col_alcaldia_hecho,
            "alcaldia_cat": col_alcaldia_cat,
            "municipio": col_municipio,
            "latitud": col_latitud,
            "longitud": col_longitud,
            "fecha_hecho": col_fecha_hecho,
            "fecha_inicio": col_fecha_inicio,
        },
    }
    save_json(fgj_meta, "fgj_metadata.json")

    total_rows = 0
    null_counts = pd.Series(0, index=cols, dtype="int64")

    delito_counter = Counter()
    categoria_counter = Counter()
    alcaldia_counter = Counter()
    anio_hecho_counter = Counter()
    anio_inicio_counter = Counter()

    agg_list = []

    chunk_iter, _ = read_csv_flexible(path, chunksize=FGJ_CHUNKSIZE, low_memory=False)

    for i, chunk in enumerate(chunk_iter, start=1):
        print(f"[FGJ] Procesando chunk {i} | filas chunk: {len(chunk):,}")

        total_rows += len(chunk)
        null_counts = null_counts.add(chunk.isna().sum(), fill_value=0)

        # Años
        if col_anio_hecho:
            chunk[col_anio_hecho] = pd.to_numeric(chunk[col_anio_hecho], errors="coerce")
            vc = chunk[col_anio_hecho].dropna().astype(int).value_counts()
            anio_hecho_counter.update(vc.to_dict())

        if col_anio_inicio:
            chunk[col_anio_inicio] = pd.to_numeric(chunk[col_anio_inicio], errors="coerce")
            vc = chunk[col_anio_inicio].dropna().astype(int).value_counts()
            anio_inicio_counter.update(vc.to_dict())

        # Delitos
        if col_delito:
            vals = chunk[col_delito].map(normalize_text).fillna("<<NA>>").value_counts()
            delito_counter.update(vals.to_dict())

        # Categorías
        if col_categoria:
            vals = chunk[col_categoria].map(normalize_text).fillna("<<NA>>").value_counts()
            categoria_counter.update(vals.to_dict())

        # Alcaldías
        alcaldia_hecho_norm = None
        alcaldia_cat_norm = None

        if col_alcaldia_hecho:
            alcaldia_hecho_norm = chunk[col_alcaldia_hecho].map(normalize_alcaldia)
        if col_alcaldia_cat:
            alcaldia_cat_norm = chunk[col_alcaldia_cat].map(normalize_alcaldia)

        if alcaldia_hecho_norm is not None and alcaldia_cat_norm is not None:
            alcaldia_final = alcaldia_hecho_norm.fillna(alcaldia_cat_norm)
        elif alcaldia_hecho_norm is not None:
            alcaldia_final = alcaldia_hecho_norm
        elif alcaldia_cat_norm is not None:
            alcaldia_final = alcaldia_cat_norm
        else:
            alcaldia_final = pd.Series([None] * len(chunk), index=chunk.index)

        vc = alcaldia_final.fillna("<<NA>>").value_counts()
        alcaldia_counter.update(vc.to_dict())

        # Agregado preliminar alcaldía-año
        if col_anio_hecho:
            tmp = pd.DataFrame({
                "anio_hecho": chunk[col_anio_hecho],
                "alcaldia_norm": alcaldia_final,
            })

            if col_delito:
                tmp["delito"] = chunk[col_delito].map(normalize_text)
            else:
                tmp["delito"] = None

            if col_categoria:
                tmp["categoria"] = chunk[col_categoria].map(normalize_text)
            else:
                tmp["categoria"] = None

            if col_latitud:
                tmp["latitud"] = chunk[col_latitud]
            else:
                tmp["latitud"] = np.nan

            if col_longitud:
                tmp["longitud"] = chunk[col_longitud]
            else:
                tmp["longitud"] = np.nan

            tmp = tmp[tmp["anio_hecho"].notna()].copy()
            if not tmp.empty:
                tmp["anio_hecho"] = tmp["anio_hecho"].astype(int)

                grp = (
                    tmp.groupby(["anio_hecho", "alcaldia_norm"], dropna=False)
                    .agg(
                        total_carpetas=("delito", "size"),
                        delitos_unicos=("delito", "nunique"),
                        categorias_unicas=("categoria", "nunique"),
                        pct_latitud_nula=("latitud", lambda s: round(s.isna().mean() * 100, 4)),
                        pct_longitud_nula=("longitud", lambda s: round(s.isna().mean() * 100, 4)),
                    )
                    .reset_index()
                )
                agg_list.append(grp)

    # Reporte global de nulos
    fgj_null_report = pd.DataFrame({
        "columna": cols,
        "nulos": [int(null_counts.get(c, 0)) for c in cols],
    })
    fgj_null_report["total_filas"] = total_rows
    fgj_null_report["pct_nulos"] = (fgj_null_report["nulos"] / total_rows * 100).round(4)
    fgj_null_report = fgj_null_report.sort_values("pct_nulos", ascending=False)
    save_csv(fgj_null_report, "fgj_null_report_global.csv")

    # Tops
    save_csv(pd.DataFrame(delito_counter.most_common(200), columns=["delito", "frecuencia"]),
             "fgj_top_delitos.csv")
    save_csv(pd.DataFrame(categoria_counter.most_common(100), columns=["categoria", "frecuencia"]),
             "fgj_top_categorias.csv")
    save_csv(pd.DataFrame(alcaldia_counter.most_common(50), columns=["alcaldia", "frecuencia"]),
             "fgj_top_alcaldias.csv")
    save_csv(pd.DataFrame(sorted(anio_hecho_counter.items()), columns=["anio_hecho", "frecuencia"]),
             "fgj_anio_hecho_counts.csv")
    save_csv(pd.DataFrame(sorted(anio_inicio_counter.items()), columns=["anio_inicio", "frecuencia"]),
             "fgj_anio_inicio_counts.csv")

    # Agregado consolidado
    if agg_list:
        fgj_agg = pd.concat(agg_list, ignore_index=True)
        fgj_agg = (
            fgj_agg.groupby(["anio_hecho", "alcaldia_norm"], dropna=False)
            .agg(
                total_carpetas=("total_carpetas", "sum"),
                delitos_unicos=("delitos_unicos", "max"),
                categorias_unicas=("categorias_unicas", "max"),
                pct_latitud_nula=("pct_latitud_nula", "mean"),
                pct_longitud_nula=("pct_longitud_nula", "mean"),
            )
            .reset_index()
            .sort_values(["anio_hecho", "alcaldia_norm"])
        )
        save_csv(fgj_agg, "fgj_agregado_alcaldia_anio.csv")
    else:
        fgj_agg = pd.DataFrame()

    resumen = {
        "total_filas_fgj": int(total_rows),
        "total_columnas_fgj": len(cols),
        "anios_hecho_detectados": sorted([int(k) for k in anio_hecho_counter.keys()]),
        "anios_inicio_detectados": sorted([int(k) for k in anio_inicio_counter.keys()]),
        "n_alcaldias_detectadas": int(len([k for k in alcaldia_counter.keys() if k != "<<NA>>"])),
    }
    save_json(resumen, "fgj_resumen.json")

    print("\nResumen FGJ:")
    print(json.dumps(resumen, ensure_ascii=False, indent=2))

    return {
        "metadata": fgj_meta,
        "null_report": fgj_null_report,
        "agregado": fgj_agg,
        "resumen": resumen,
    }


# =========================================================
# 4) ANÁLISIS POBREZA
# =========================================================

def analyze_pobreza(path):
    print("\n" + "=" * 90)
    print("ANÁLISIS POBREZA")
    print("=" * 90)

    pobreza, enc = read_csv_flexible(path, low_memory=False)

    cols = pobreza.columns.tolist()

    # Resolver columnas
    col_anio = resolve_column(cols, ["año", "anio"])
    col_municipio = resolve_column(cols, ["municipio"])
    col_entidad = resolve_column(cols, ["entidad"])
    col_factor = resolve_column(cols, ["factor"])

    cont_candidates = {
        "mmip": resolve_column(cols, ["mmip"]),
        "rei": resolve_column(cols, ["rei"]),
        "nbi": resolve_column(cols, ["nbi"]),
        "CBDj": resolve_column(cols, ["CBDj", "cbdj"]),
        "CSj": resolve_column(cols, ["CSj", "csj"]),
        "CTEU": resolve_column(cols, ["CTEU", "cteu"]),
        "CENJ": resolve_column(cols, ["CENJ", "cenj"]),
        "ett": resolve_column(cols, ["ett"]),
        "CASi": resolve_column(cols, ["CASi", "casi"]),
        "CASSi": resolve_column(cols, ["CASSi", "cassi"]),
        "cyt": resolve_column(cols, ["cyt"]),
        "ccevj": resolve_column(cols, ["ccevj", "ccevj"]),
    }

    estr_candidates = {
        "E_mmip": resolve_column(cols, ["E_mmip", "e_mmip"]),
        "E_rei": resolve_column(cols, ["E_rei", "e_rei"]),
        "E_nbi": resolve_column(cols, ["E_nbi", "e_nbi"]),
        "E_CBDj": resolve_column(cols, ["E_CBDj", "e_cbdj"]),
        "E_CSj": resolve_column(cols, ["E_CSj", "e_csj"]),
        "E_CTELJ": resolve_column(cols, ["E_CTELJ", "e_ctelj"]),
        "E_CENJ": resolve_column(cols, ["E_CENJ", "e_cenj"]),
        "E_CASSi": resolve_column(cols, ["E_CASSi", "e_cassi"]),
        "E_CASi": resolve_column(cols, ["E_CASi", "e_casi"]),
        "E_cict": resolve_column(cols, ["E_cict", "e_cict"]),
        "E_ett": resolve_column(cols, ["E_ett", "e_ett"]),
        "E_cyt": resolve_column(cols, ["E_cyt", "e_cyt"]),
        "E_ccevj": resolve_column(cols, ["E_ccevj", "e_ccevj"]),
    }

    pobreza_meta = {
        "encoding_usado": enc,
        "n_filas": int(len(pobreza)),
        "n_columnas": int(pobreza.shape[1]),
        "columnas": cols,
        "columnas_resueltas": {
            "año": col_anio,
            "municipio": col_municipio,
            "entidad": col_entidad,
            "factor": col_factor,
            "continuas": cont_candidates,
            "estratos": estr_candidates,
        },
    }
    save_json(pobreza_meta, "pobreza_metadata.json")

    # Nulos
    pobreza_null_report = pd.DataFrame({
        "columna": cols,
        "dtype": pobreza.dtypes.astype(str).values,
        "nulos": pobreza.isna().sum().values,
        "pct_nulos": (pobreza.isna().sum().values / len(pobreza) * 100).round(4),
        "n_unicos_sin_na": [pobreza[c].nunique(dropna=True) for c in cols],
    }).sort_values("pct_nulos", ascending=False)
    save_csv(pobreza_null_report, "pobreza_null_report_global.csv")

    # Años
    if col_anio:
        pobreza[col_anio] = pd.to_numeric(pobreza[col_anio], errors="coerce")
        anios_counts = (
            pobreza[col_anio]
            .dropna()
            .astype(int)
            .value_counts()
            .sort_index()
            .reset_index()
        )
        anios_counts.columns = ["anio", "frecuencia"]
    else:
        anios_counts = pd.DataFrame(columns=["anio", "frecuencia"])
    save_csv(anios_counts, "pobreza_anio_counts.csv")

    # Municipios
    if col_municipio:
        municipios_counts = (
            pobreza[col_municipio]
            .value_counts(dropna=False)
            .reset_index()
        )
        municipios_counts.columns = ["municipio", "frecuencia"]
    else:
        municipios_counts = pd.DataFrame(columns=["municipio", "frecuencia"])
    save_csv(municipios_counts, "pobreza_municipio_counts.csv")

    # Detectar si los municipios parecen ser CDMX
    if col_municipio:
        pobreza["municipio_num"] = pd.to_numeric(pobreza[col_municipio], errors="coerce")
        unique_municipios = sorted([int(x) for x in pobreza["municipio_num"].dropna().unique()])
    else:
        pobreza["municipio_num"] = np.nan
        unique_municipios = []

    municipios_subset_cdmx = set(unique_municipios).issubset(set(MAP_MUNICIPIO_CDMX.keys())) and len(unique_municipios) > 0

    if municipios_subset_cdmx:
        pobreza_cdmx = pobreza.copy()
        pobreza_cdmx["alcaldia_norm"] = pobreza_cdmx["municipio_num"].map(MAP_MUNICIPIO_CDMX).map(normalize_alcaldia)
        filtro_usado = "municipio ya corresponde a alcaldías de CDMX"
    else:
        # Si no coincide con el mapa completo, intentamos mapear lo que sí pueda
        pobreza_cdmx = pobreza.copy()
        pobreza_cdmx["alcaldia_norm"] = pobreza_cdmx["municipio_num"].map(MAP_MUNICIPIO_CDMX).map(normalize_alcaldia)
        filtro_usado = "mapeo parcial por municipio; revisar manualmente si la cobertura no es completa"

    # Continuas detectadas
    cont_detectadas = {k: v for k, v in cont_candidates.items() if v is not None}
    estr_detectadas = {k: v for k, v in estr_candidates.items() if v is not None}

    # Describe de continuas
    if cont_detectadas:
        cont_cols = list(cont_detectadas.values())
        pobreza_desc = pobreza_cdmx[cont_cols].apply(pd.to_numeric, errors="coerce").describe().T.reset_index()
        pobreza_desc.rename(columns={"index": "variable"}, inplace=True)
    else:
        pobreza_desc = pd.DataFrame()
    save_csv(pobreza_desc, "pobreza_describe_continuas.csv")

    # Agregado alcaldía-año
    if col_anio and col_factor:
        rows = []
        pobreza_cdmx[col_factor] = pd.to_numeric(pobreza_cdmx[col_factor], errors="coerce")

        for keys, sub in pobreza_cdmx.groupby([col_anio, "alcaldia_norm"], dropna=False):
            anio, alcaldia = keys
            row = {
                "anio": int(anio) if pd.notna(anio) else np.nan,
                "alcaldia_norm": alcaldia,
                "n_registros": int(len(sub)),
                "factor_sum": float(sub[col_factor].sum(skipna=True)) if pd.notna(sub[col_factor].sum(skipna=True)) else np.nan,
            }

            for nombre_logico, col_real in cont_detectadas.items():
                row[f"{nombre_logico}_wmean"] = weighted_mean(
                    pd.to_numeric(sub[col_real], errors="coerce"),
                    sub[col_factor]
                )

            for nombre_logico, col_real in estr_detectadas.items():
                row[f"{nombre_logico}_wmode"] = weighted_mode(sub[col_real], sub[col_factor])

            rows.append(row)

        pobreza_agg = pd.DataFrame(rows).sort_values(["anio", "alcaldia_norm"]).reset_index(drop=True)
    else:
        pobreza_agg = pd.DataFrame()

    save_csv(pobreza_agg, "pobreza_agregado_alcaldia_anio.csv")

    resumen = {
        "total_filas_pobreza": int(len(pobreza)),
        "total_columnas_pobreza": int(pobreza.shape[1]),
        "encoding_usado": enc,
        "anios_detectados": sorted(anios_counts["anio"].astype(int).tolist()) if not anios_counts.empty else [],
        "municipios_detectados": unique_municipios,
        "municipios_parecen_cdmx": bool(municipios_subset_cdmx),
        "criterio_mapeo_alcaldia": filtro_usado,
        "n_alcaldias_mapeadas": int(pobreza_cdmx["alcaldia_norm"].nunique(dropna=True)),
        "variables_continuas_detectadas": list(cont_detectadas.keys()),
        "variables_estratos_detectadas": list(estr_detectadas.keys()),
    }
    save_json(resumen, "pobreza_resumen.json")

    print("\nResumen pobreza:")
    print(json.dumps(resumen, ensure_ascii=False, indent=2))

    return {
        "metadata": pobreza_meta,
        "null_report": pobreza_null_report,
        "agregado": pobreza_agg,
        "resumen": resumen,
    }


# =========================================================
# 5) CRUCE PRELIMINAR Y TEMPORALIDAD
# =========================================================

def preliminary_join_and_temporal_assessment(fgj_agg, pobreza_agg):
    print("\n" + "=" * 90)
    print("CRUCE PRELIMINAR Y EVALUACIÓN DE TEMPORALIDAD")
    print("=" * 90)

    if fgj_agg.empty or pobreza_agg.empty:
        print("No se puede realizar el cruce preliminar porque uno de los agregados está vacío.")
        return {
            "common_years": [],
            "common_alcaldias": [],
            "main_window": [],
            "lag_window": [],
            "joined": pd.DataFrame(),
        }

    # Normalizar nombres por seguridad
    fgj = fgj_agg.copy()
    pob = pobreza_agg.copy()

    fgj["alcaldia_norm"] = fgj["alcaldia_norm"].map(normalize_alcaldia)
    pob["alcaldia_norm"] = pob["alcaldia_norm"].map(normalize_alcaldia)

    fgj["anio"] = pd.to_numeric(fgj["anio_hecho"], errors="coerce").astype("Int64")
    pob["anio"] = pd.to_numeric(pob["anio"], errors="coerce").astype("Int64")

    # Años detectados
    fgj_years = sorted([int(x) for x in fgj["anio"].dropna().unique()])
    pob_years = sorted([int(x) for x in pob["anio"].dropna().unique()])
    common_years = sorted(set(fgj_years).intersection(set(pob_years)))

    # Alcaldías detectadas
    fgj_alcaldias = sorted([x for x in fgj["alcaldia_norm"].dropna().unique()])
    pob_alcaldias = sorted([x for x in pob["alcaldia_norm"].dropna().unique()])
    common_alcaldias = sorted(set(fgj_alcaldias).intersection(set(pob_alcaldias)))

    # Join contemporáneo
    joined_common = pd.merge(
        pob[pob["anio"].isin(common_years)],
        fgj[fgj["anio"].isin(common_years)],
        on=["anio", "alcaldia_norm"],
        how="inner",
    )
    save_csv(joined_common, "join_preliminar_alcaldia_anio_contemporaneo.csv")

    # Join rezagado: pobreza 2016-2020 contra FGJ posterior al último año común
    lag_years = [y for y in fgj_years if len(common_years) > 0 and y > max(common_years)]

    # Para rezago, se promedian indicadores sociales sobre años comunes y se comparan con años posteriores
    if common_years:
        pob_common = pob[pob["anio"].isin(common_years)].copy()

        # seleccionar columnas numéricas útiles
        social_cols = [
            c for c in pob_common.columns
            if c.endswith("_wmean") or c.endswith("_wmode")
        ]

        pob_social_base = (
            pob_common.groupby("alcaldia_norm", dropna=False)[social_cols]
            .mean(numeric_only=True)
            .reset_index()
        )
        save_csv(pob_social_base, "pobreza_base_social_promedio_ventana_comun.csv")

        fgj_lag = fgj[fgj["anio"].isin(lag_years)].copy()
        join_lag = pd.merge(
            fgj_lag,
            pob_social_base,
            on="alcaldia_norm",
            how="left"
        )
    else:
        pob_social_base = pd.DataFrame()
        join_lag = pd.DataFrame()

    save_csv(join_lag, "join_preliminar_rezagado.csv")

    resumen_temporal = {
        "fgj_years": fgj_years,
        "pobreza_years": pob_years,
        "common_years": common_years,
        "lag_years": lag_years,
        "fgj_alcaldias_detectadas": fgj_alcaldias,
        "pobreza_alcaldias_detectadas": pob_alcaldias,
        "common_alcaldias": common_alcaldias,
        "n_obs_join_contemporaneo": int(len(joined_common)),
        "n_obs_join_rezagado": int(len(join_lag)),
    }
    save_json(resumen_temporal, "evaluacion_temporalidad.json")

    print("\nResumen temporalidad:")
    print(json.dumps(resumen_temporal, ensure_ascii=False, indent=2))

    return {
        "common_years": common_years,
        "common_alcaldias": common_alcaldias,
        "main_window": common_years,
        "lag_window": lag_years,
        "joined": joined_common,
        "joined_lag": join_lag,
        "summary": resumen_temporal,
    }


# =========================================================
# 6) RESUMEN FINAL DEL EDA
# =========================================================

def build_final_summary(fgj_result, pobreza_result, temporal_result):
    top_fgj_nulls = fgj_result["null_report"].head(10)[["columna", "pct_nulos"]]
    top_pob_nulls = pobreza_result["null_report"].head(10)[["columna", "pct_nulos"]]

    top_delitos_path = OUT_DIR / "fgj_top_delitos.csv"
    if top_delitos_path.exists():
        top_delitos = pd.read_csv(top_delitos_path).head(10)
    else:
        top_delitos = pd.DataFrame(columns=["delito", "frecuencia"])

    top_cat_path = OUT_DIR / "fgj_top_categorias.csv"
    if top_cat_path.exists():
        top_categorias = pd.read_csv(top_cat_path).head(10)
    else:
        top_categorias = pd.DataFrame(columns=["categoria", "frecuencia"])

    common_years = temporal_result["common_years"]
    lag_years = temporal_result["lag_window"]

    if len(common_years) >= 3:
        recomendacion_temporal = (
            f"Ventana principal recomendada: análisis contemporáneo por alcaldía-año en {common_years}. "
            f"Ventana secundaria recomendada: análisis rezagado usando pobreza promedio de {common_years} "
            f"para contrastarla con FGJ en {lag_years}."
        )
    elif len(common_years) > 0:
        recomendacion_temporal = (
            f"Hay pocos años comunes ({common_years}). Conviene usar esos años como análisis principal "
            f"y tratar el rezago hacia {lag_years} como exploración secundaria."
        )
    else:
        recomendacion_temporal = (
            "No se detectaron años comunes entre ambas bases para análisis contemporáneo. "
            "En ese caso, el estudio debe plantearse solo como análisis rezagado."
        )

    resumen_txt = []
    resumen_txt.append("RESUMEN FINAL DEL ANÁLISIS EXPLORATORIO\n")
    resumen_txt.append("=" * 70 + "\n\n")

    resumen_txt.append("1) FGJ\n")
    resumen_txt.append(f"- Filas detectadas: {fgj_result['resumen']['total_filas_fgj']:,}\n")
    resumen_txt.append(f"- Columnas detectadas: {fgj_result['resumen']['total_columnas_fgj']}\n")
    resumen_txt.append(f"- Años de hecho detectados: {fgj_result['resumen']['anios_hecho_detectados']}\n")
    resumen_txt.append(f"- Años de inicio detectados: {fgj_result['resumen']['anios_inicio_detectados']}\n")
    resumen_txt.append(f"- Número de alcaldías detectadas: {fgj_result['resumen']['n_alcaldias_detectadas']}\n\n")

    resumen_txt.append("Top 10 columnas con mayor porcentaje de nulos en FGJ:\n")
    for _, row in top_fgj_nulls.iterrows():
        resumen_txt.append(f"  - {row['columna']}: {row['pct_nulos']}%\n")
    resumen_txt.append("\n")

    resumen_txt.append("Top 10 delitos más frecuentes en FGJ:\n")
    for _, row in top_delitos.iterrows():
        resumen_txt.append(f"  - {row['delito']}: {row['frecuencia']}\n")
    resumen_txt.append("\n")

    resumen_txt.append("Top 10 categorías más frecuentes en FGJ:\n")
    for _, row in top_categorias.iterrows():
        resumen_txt.append(f"  - {row['categoria']}: {row['frecuencia']}\n")
    resumen_txt.append("\n")

    resumen_txt.append("2) POBREZA MULTIDIMENSIONAL\n")
    resumen_txt.append(f"- Filas detectadas: {pobreza_result['resumen']['total_filas_pobreza']:,}\n")
    resumen_txt.append(f"- Columnas detectadas: {pobreza_result['resumen']['total_columnas_pobreza']}\n")
    resumen_txt.append(f"- Encoding usado: {pobreza_result['resumen']['encoding_usado']}\n")
    resumen_txt.append(f"- Años detectados: {pobreza_result['resumen']['anios_detectados']}\n")
    resumen_txt.append(f"- Municipios detectados: {pobreza_result['resumen']['municipios_detectados']}\n")
    resumen_txt.append(f"- ¿Municipios parecen CDMX?: {pobreza_result['resumen']['municipios_parecen_cdmx']}\n")
    resumen_txt.append(f"- Alcaldías mapeadas: {pobreza_result['resumen']['n_alcaldias_mapeadas']}\n")
    resumen_txt.append(f"- Variables continuas detectadas: {pobreza_result['resumen']['variables_continuas_detectadas']}\n")
    resumen_txt.append(f"- Variables de estrato detectadas: {pobreza_result['resumen']['variables_estratos_detectadas']}\n\n")

    resumen_txt.append("Top 10 columnas con mayor porcentaje de nulos en pobreza:\n")
    for _, row in top_pob_nulls.iterrows():
        resumen_txt.append(f"  - {row['columna']}: {row['pct_nulos']}%\n")
    resumen_txt.append("\n")

    resumen_txt.append("3) CRUCE Y TEMPORALIDAD\n")
    resumen_txt.append(f"- Años comunes detectados: {temporal_result['common_years']}\n")
    resumen_txt.append(f"- Años posteriores para análisis rezagado: {temporal_result['lag_window']}\n")
    resumen_txt.append(f"- Alcaldías comunes detectadas: {temporal_result['common_alcaldias']}\n")
    resumen_txt.append(f"- Observaciones del join contemporáneo: {len(temporal_result['joined'])}\n")
    resumen_txt.append(f"- Observaciones del join rezagado: {len(temporal_result['joined_lag'])}\n\n")

    resumen_txt.append("4) RECOMENDACIÓN METODOLÓGICA PRELIMINAR\n")
    resumen_txt.append(recomendacion_temporal + "\n")
    resumen_txt.append(
        "Además, se recomienda que el objetivo principal sea explicativo-territorial y que la predicción "
        "se deje como complemento. También se recomienda seleccionar un subconjunto de delitos socialmente "
        "relevantes antes del modelado final.\n"
    )

    resumen_final = "".join(resumen_txt)
    save_txt(resumen_final, "resumen_final_eda.txt")
    print("\n" + "=" * 90)
    print(resumen_final)
    print("=" * 90)

    return resumen_final


# =========================================================
# 7) EJECUCIÓN COMPLETA
# =========================================================

def run_eda():
    if not FGJ_FILE.exists():
        raise FileNotFoundError(f"No se encontró el archivo FGJ: {FGJ_FILE}")
    if not POBREZA_FILE.exists():
        raise FileNotFoundError(f"No se encontró el archivo de pobreza: {POBREZA_FILE}")

    fgj_result = analyze_fgj(FGJ_FILE)
    pobreza_result = analyze_pobreza(POBREZA_FILE)
    temporal_result = preliminary_join_and_temporal_assessment(
        fgj_result["agregado"],
        pobreza_result["agregado"]
    )
    build_final_summary(fgj_result, pobreza_result, temporal_result)

    print(f"\nEDA completado. Resultados guardados en: {OUT_DIR}")


run_eda()

BASE_DIR: d:\DMProject
FGJ_FILE: d:\DMProject\datasets\carpetasFGJ_acumulado_2025_01.csv | Existe: True
POBREZA_FILE: d:\DMProject\datasets\enigh_16_20.csv | Existe: True
OUT_DIR: d:\DMProject\Documentation\eda_outputs

ANÁLISIS FGJ
[OK] Archivo leído con encoding: utf-8
[OK] Guardado JSON: d:\DMProject\Documentation\eda_outputs\fgj_metadata.json
[OK] Archivo leído con encoding: utf-8
[FGJ] Procesando chunk 1 | filas chunk: 200,000
[FGJ] Procesando chunk 2 | filas chunk: 200,000
[FGJ] Procesando chunk 3 | filas chunk: 200,000
[FGJ] Procesando chunk 4 | filas chunk: 200,000
[FGJ] Procesando chunk 5 | filas chunk: 200,000
[FGJ] Procesando chunk 6 | filas chunk: 200,000
[FGJ] Procesando chunk 7 | filas chunk: 200,000
[FGJ] Procesando chunk 8 | filas chunk: 200,000
[FGJ] Procesando chunk 9 | filas chunk: 200,000
[FGJ] Procesando chunk 10 | filas chunk: 200,000
[FGJ] Procesando chunk 11 | filas chunk: 98,743
[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\fgj_null_report_global.cs